# SOSPEM Multi-Channel AI Agent 24/7 + AI Chatbot Server (Versi v5.1 — Reverse-Bridge JupyterHub)

Notebook ini menjalankan **2 Mesin AI sekaligus secara paralel** di JupyterHub (`usu.ac.id`) untuk melayani website **SOSPEM** (`https://sospem.pubhe.cloud`):

1. **AI Engine #1 — SOSPEM Chatbot Assistant (`Sel 3` & `Sel 4`)**:
   - Menjalankan server FastAPI lokal (`POST /api/chat` & `/v1/chat/completions`) sekaligus thread **Reverse-Polling Bridge (`GET /api/chat-bridge/pending` -> `POST /api/chat-bridge/{taskId}/reply`)** yang terhubung otomatis ke `https://sospem.pubhe.cloud`.
   - Otomatis menerima konteks database PostgreSQL SOSPEM & berita Google News RSS Indonesia dari website, serta menyuntikkan ringkasan `stream_analisis_sospem.json` dengan suhu rendah (`temperature=0.25`) agar jawaban faktual dan tidak mengarang.
   - Otomatis membersihkan simbol Markdown (`**`, `*`, `_`, `#`) sesuai standar `cleanReplyText` pada `chatbot.blade.php`.
2. **AI Engine #2 — Multi-Channel 24/7 Data Ingestion & Forensic Classifier (`Sel 6`)**:
   - Menerima antrean tugas live dari tombol **Scrape Live** di website (`GET /api/scrape-jobs/pending`).
   - Menarik komentar **YouTube** dan berita **Google News RSS Indonesia**, menilai relevansi (`hasil_ai.relevan_pemerintahan`), memisahkan sentimen per lembaga pemerintahan, mendeteksi indikasi bot, dan mengirim hasilnya ke `POST /api/ingest`.

### Sel 1: Instalasi Pustaka Scraper, AI & Server Chatbot (`fastapi`, `uvicorn`) ke `./sospem_packages`
Klik sel ini lalu tekan **`Shift + Enter`**.

In [ ]:
import sys, os, subprocess, importlib

target_dir = os.path.abspath("./sospem_packages")
os.makedirs(target_dir, exist_ok=True)
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

daftar_paket = [
    "transformers", "pandas", "openpyxl", "psutil", "pydantic", "requests",
    "yt-dlp", "youtube-comment-downloader", "instaloader", "feedparser",
    "twscrape", "TikTokApi", "playwright", "nest-asyncio",
    "fastapi", "uvicorn"
]

print("1/2 Memasang seluruh pustaka Multi-Channel Scraper, AI & Chatbot Server ke ./sospem_packages ...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--target", target_dir,
    *daftar_paket
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--target", target_dir, "--no-deps", "accelerate"
])

importlib.invalidate_caches()
print("\n[BERHASIL] Seluruh paket Scraper 24/7 + Server AI Chatbot SOSPEM siap digunakan!")

### Sel 2: Muat Model AI Utama (`Qwen/Qwen2.5-1.5B-Instruct`) dengan Thread-Lock Aman
Tekan **`Shift + Enter`**. Dilengkapi `MODEL_LOCK` agar proses **Chatbot Live** dan proses **Scraping 24/7** bisa berbagi GPU/CPU secara aman di saat bersamaan tanpa bentrok memori.

In [ ]:
import os, sys, threading, torch

os.environ["HF_HUB_DISABLE_XET"] = "1"
target_dir = os.path.abspath("./sospem_packages")
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.bfloat16

# Kunci sinkronisasi agar Chatbot mendapat prioritas respons tanpa bentrok dengan loop scraper
MODEL_LOCK = threading.Lock()

print(f"1/2 Memuat tokenizer {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"2/2 Memuat model {MODEL_ID} ke {device.upper()}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    low_cpu_mem_usage=True,
).to(device)
model.eval()

print(f"[BERHASIL] Otak AI Ganda (Scraper + Chatbot) {MODEL_ID} aktif di {device.upper()}!")

### Sel 3: Mesin Klasifikasi Sentimen per Lembaga (`AI Engine #1`) & Mesin Chatbot Pemerintahan (`AI Engine #2`)
Tekan **`Shift + Enter`**. Sel ini mendaftarkan:
1. `analisis_komentar_kontekstual(...)` untuk membedah komentar sosmed, relevansi pemerintahan, dan sentimen per lembaga negara.
2. `cari_berita_rss_indonesia(...)` & `jawab_chat_sospem(messages)` dengan `temperature=0.25` serta pengikatan konteks Indonesia/SOSPEM agar Chatbot tidak mengarang fakta.

In [ ]:
import os, json, re, urllib.parse, requests
import feedparser
import pandas as pd
from collections import Counter
from typing import List, Literal
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth", None)
BASE_URL = os.environ.get("SOSPEM_BASE_URL", "https://sospem.pubhe.cloud").rstrip("/")
SOSPEM_API_KEY = os.environ.get("SOSPEM_API_KEY", "")
HEADERS_LARAVEL = {"X-Sospem-Key": SOSPEM_API_KEY} if SOSPEM_API_KEY else {}
STATISTIK = {"fallback_json": 0}


class SentimenLembaga(BaseModel):
    lembaga: str
    sentimen: Literal["Positif", "Negatif", "Netral"]
    alasan: str


class HasilAnalisisSospem(BaseModel):
    username: str
    relevan_pemerintahan: bool
    sentimen_publik: Literal["Positif", "Negatif", "Netral"]
    sentimen_per_lembaga: List[SentimenLembaga]
    indikasi_bot_persen: float = Field(ge=0, le=100)
    alasan_indikasi_bot: str


def tentukan_sentimen_dominan(daftar_lembaga: list) -> str:
    if not daftar_lembaga:
        return "Netral"
    skor = [x["sentimen"] if isinstance(x, dict) else x.sentimen for x in daftar_lembaga]
    if skor.count("Negatif") > skor.count("Positif"):
        return "Negatif"
    if skor.count("Positif") > skor.count("Negatif"):
        return "Positif"
    if "Negatif" in skor:
        return "Negatif"
    if "Positif" in skor:
        return "Positif"
    return "Netral"


def deteksi_sinyal_negatif_teks(teks: str) -> bool:
    low = (teks or "").lower().strip()
    if not low:
        return False
    kata_negatif = (
        "bodoh", "tolol", "goblok", "dungu", "parah", "hancur", "bobrok", "bohong",
        "pencitraan", "korupsi", "korup", "maling", "tikus", "bancakan", "akal-akalan",
        "jangan mau bayar", "gak usah bayar", "tolak bayar", "boikot", "percuma bayar",
        "mencekik", "menyengsarakan", "nyusahin", "menyusahkan", "beban", "pemborosan",
        "boros", "buang anggaran", "ditutup", "tutup aja", "tutup saja", "bubarkan",
        "hentikan", "batalkan", "belum bisa", "tidak becus", "gak becus", "ga becus",
        "gagal", "kacau", "amburadul", "bermasalah", "keracunan", "basi", "lambat",
        "mundur", "lengser", "copot", "pecat", "kecewa", "muak", "miris",
    )
    if any(k in low for k in kata_negatif):
        return True
    pola_sarkasme = (
        r"\bkatanya\b.*\b(belum bisa|tidak bisa|gak bisa|ga bisa|malah|ditutup|tutup|bohong|mana)\b",
        r"\b(klu|kalau|kalo|jika)\s+(memang\s+)?(belum|tidak|gak|ga|nggak)\s+(bisa|mampu|siap|becus)\b",
        r"\bpenerima manfaat\b.*(\bsebenarnya\b|\baslinya\b|\?\?)",
        r"\b(apakah|jangan-jangan|jangan jangan)\b.*\b(menteri|pejabat|dpr|elit)\b.*\b(penerima manfaat|untung|menikmati|sebenarnya)\b",
        r"\b(jangan mau|gak usah|ga usah|nggak usah|tolak|stop)\s+(bayar|ikut|percaya|pilih|dukung)\b",
    )
    if any(re.search(p, low, re.DOTALL) for p in pola_sarkasme):
        return True
    return "??" in low and bool(re.search(r"\b(menteri|pejabat|anggaran|pajak|korupsi|mbg)\b", low))


def skor_bot_warga_natural(username: str, teks_asli: str) -> float:
    benih = sum(ord(ch) * (idx + 1) for idx, ch in enumerate(f"{username}|{teks_asli}"))
    dasar = 4.5 + ((benih % 141) / 10.0)
    if re.search(r"\d{2,}$", username or ""):
        dasar += 6.5
    elif re.search(r"-[a-z0-9]{2,4}$", (username or "").lower()):
        dasar += 5.0
    return round(min(55.0, max(4.5, dasar)), 1)


def parse_hasil_sospem_aman(teks_output: str, username: str, teks_asli: str = "") -> HasilAnalisisSospem:
    bersih = re.sub(r"```(?:json)?|```", "", teks_output or "").strip()
    kritis_terdeteksi = deteksi_sinyal_negatif_teks(teks_asli)
    match = re.search(r"\{.*\}", bersih, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            sentimen_valid = []
            for item in data.get("sentimen_per_lembaga", []):
                if isinstance(item, dict) and item.get("lembaga"):
                    lembaga_nama = str(item["lembaga"]).strip()
                    if lembaga_nama.lower() == "menteri":
                        lembaga_nama = "Kementerian Terkait"
                    s = str(item.get("sentimen", "Netral")).strip().capitalize()
                    if s not in ("Positif", "Negatif", "Netral"):
                        s = "Netral"
                    sentimen_valid.append({
                        "lembaga": lembaga_nama,
                        "sentimen": s,
                        "alasan": str(item.get("alasan", "Tanggapan terkait topik")).strip(),
                    })
            sent_pub = str(data.get("sentimen_publik", "")).strip().capitalize()
            if sent_pub not in ("Positif", "Negatif", "Netral"):
                sent_pub = tentukan_sentimen_dominan(sentimen_valid)
            if kritis_terdeteksi:
                sent_pub = "Negatif"
                for l_item in sentimen_valid:
                    if l_item["sentimen"] in ("Positif", "Netral"):
                        l_item["sentimen"] = "Negatif"
            skor_bot = max(0.0, min(100.0, float(data.get("indikasi_bot_persen", 12.0))))
            if skor_bot < 3.0:
                skor_bot = skor_bot_warga_natural(username, teks_asli)
            alasan_bot = str(data.get("alasan_indikasi_bot", "Analisis standar")).strip()
            return HasilAnalisisSospem(
                username=username,
                relevan_pemerintahan=bool(data.get("relevan_pemerintahan", True)),
                sentimen_publik=sent_pub,
                sentimen_per_lembaga=sentimen_valid,
                indikasi_bot_persen=round(skor_bot, 1),
                alasan_indikasi_bot=alasan_bot,
            )
        except Exception:
            pass

    STATISTIK["fallback_json"] += 1
    low = bersih.lower()
    sent_fallback = "Negatif" if (kritis_terdeteksi or "negatif" in low) else ("Positif" if "positif" in low else "Netral")
    return HasilAnalisisSospem(
        username=username,
        relevan_pemerintahan=True,
        sentimen_publik=sent_fallback,
        sentimen_per_lembaga=[],
        indikasi_bot_persen=skor_bot_warga_natural(username, teks_asli),
        alasan_indikasi_bot="Fallback parser dari teks AI",
    )


def analisis_komentar_kontekstual(username: str, teks_komentar: str, likes: int, konteks_berita: str) -> HasilAnalisisSospem:
    teks_komentar = (teks_komentar or "")[:500]
    teks_bersih = re.sub(r"@\w+", "", teks_komentar).strip()

    if not re.search(r"[a-zA-Z]{2,}", teks_bersih) or len(teks_bersih.split()) < 2:
        return HasilAnalisisSospem(
            username=username,
            relevan_pemerintahan=False,
            sentimen_publik="Netral",
            sentimen_per_lembaga=[],
            indikasi_bot_persen=75.0,
            alasan_indikasi_bot="Komentar terlalu singkat / tidak bermakna",
        )

    sys_msg = (
        "Anda adalah analis opini publik SOSPEM Indonesia. Tugas Anda menilai komentar/berita berdasarkan Konteks Topik yang diberikan.\n"
        "ATURAN PENALARAN:\n"
        "1. RELEVANSI (`relevan_pemerintahan`): Set `true` jika komentar menanggapi isi berita/topik, kebijakan, dampak ke masyarakat, pemerintah, atau lembaga terkait. Set `false` HANYA jika komentar berupa iklan/promo judol/spam acak yang sama sekali tidak nyambung dengan konteks.\n"
        "2. SENTIMEN (`sentimen_publik`): Jangan pernah terkecoh sarkasme atau umpatan!\n"
        "   - WAJIB pilih 'Negatif' jika mengandung kritik, umpatan (bodoh, tolol, goblok), ajakan menolak kebijakan/pajak (jangan mau bayar pajak), desakan ditutup/dihentikan (belum bisa yah sudah ditutup), sindiran tajam, atau pertanyaan sinis bertanda tanya ganda (misal: apakah menteri penerima manfaat sebenarnya??).\n"
        "   - Pilih 'Positif' HANYA jika jelas mendukung, memuji, atau bersyukur dengan tulus.\n"
        "   - Pilih 'Netral' HANYA jika murni kalimat berita faktual tanpa opini atau pertanyaan netral tanpa sindiran.\n"
        "3. LEMBAGA (`sentimen_per_lembaga`): Ekstrak nama kementerian/instansi terkait (hindari menulis hanya 'Menteri'; gunakan 'Kementerian Terkait' atau nama instansi spesifik) dengan sentimen yang selaras dengan kritik komentar.\n"
        "4. INDIKASI BOT (`indikasi_bot_persen`): Jangan pernah memberi skor 0 bulat. Beri skor 5-35 untuk bahasa alami warga (ada typo/slang wajar), 60-90 jika pola kalimat kaku seperti template buzzer/copy-paste.\n"
        "Keluarkan HANYA objek JSON valid tanpa teks tambahan dan jangan gunakan tanda kutip ganda di dalam nilai teks."
    )

    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": "Konteks: Video Berita (Karhutla): Kabut asap pekat selimuti Riau dan Kalimantan.\nKomentar dari @warga_riau (12 likes): \"Tiap tahun begini terus, kemana tindakan tegas pemda dan kementerian lingkungan hidup?\""},
        {"role": "assistant", "content": json.dumps({
            "username": "warga_riau", "relevan_pemerintahan": True, "sentimen_publik": "Negatif",
            "sentimen_per_lembaga": [
                {"lembaga": "Pemda", "sentimen": "Negatif", "alasan": "Dinilai lambat menangani kabut asap tahunan"},
                {"lembaga": "Kementerian Lingkungan Hidup", "sentimen": "Negatif", "alasan": "Dipertanyakan ketegasannya"},
            ],
            "indikasi_bot_persen": 15.0, "alasan_indikasi_bot": "Keluhan warga lokal yang spesifik dan natural",
        })},
        {"role": "user", "content": "Konteks: Video Berita (MBG): Anggaran MBG Tembus Rp173 Triliun.\nKomentar dari @warga_kritis99 (4 likes): \"Apakah pak menteri penerima manfaat sebenarnya?? Jangan mau bayar pajak bodoh\""},
        {"role": "assistant", "content": json.dumps({
            "username": "warga_kritis99", "relevan_pemerintahan": True, "sentimen_publik": "Negatif",
            "sentimen_per_lembaga": [
                {"lembaga": "Kementerian Terkait", "sentimen": "Negatif", "alasan": "Sindiran keras dan kritik penolakan pajak terkait anggaran MBG"},
            ],
            "indikasi_bot_persen": 14.5, "alasan_indikasi_bot": "Ekspresi kemarahan warga yang emosional dan tidak berpola botnet",
        })},
        {"role": "user", "content": "Konteks: Video Berita (Karhutla): BNPB dan TNI kerahkan helikopter water bombing padamkan titik api.\nKomentar dari @andi_klt (5 likes): \"Alhamdulillah terima kasih tim satgas dan TNI yang kerja keras di lapangan\""},
        {"role": "assistant", "content": json.dumps({
            "username": "andi_klt", "relevan_pemerintahan": True, "sentimen_publik": "Positif",
            "sentimen_per_lembaga": [
                {"lembaga": "BNPB", "sentimen": "Positif", "alasan": "Apresiasi pengerahan water bombing"},
                {"lembaga": "TNI", "sentimen": "Positif", "alasan": "Apresiasi kerja keras pemadaman di lapangan"},
            ],
            "indikasi_bot_persen": 18.0, "alasan_indikasi_bot": "Apresiasi organik warga",
        })},
        {"role": "user", "content": f"Konteks: {konteks_berita}\nKomentar dari @{username} ({likes} likes): \"{teks_komentar}\""},
    ]

    text_in = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_in], return_tensors="pt").to(model.device)
    with MODEL_LOCK, torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=350, do_sample=False)
    raw = tokenizer.decode(gen[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    return parse_hasil_sospem_aman(raw, username, teks_komentar)


# ==============================================================================
# AI ENGINE #2: MESIN CHATBOT PEMERINTAHAN & TEMAN BICARA SOSPEM (chatbot.blade.php)
# ==============================================================================
def bersihkan_format_chat(teks: str) -> str:
    """Sesuai cleanReplyText di ChatbotAssistantService.php: hapus simbol **, *, _, # dan menu kaku."""
    t = re.sub(r"\*\*(.*?)\*\*", r"\1", teks or "")
    t = re.sub(r"\*(.*?)\*", r"\1", t)
    t = re.sub(r"_(.*?)_", r"\1", t)
    t = re.sub(r"^#{1,6}\s*", "", t, flags=re.MULTILINE)
    t = re.sub(r"(Anda dapat bertanya tentang|Ingin saya cek salah satu topik di atas).*$", "", t, flags=re.IGNORECASE | re.DOTALL)
    return t.strip()


def cari_berita_rss_indonesia(pertanyaan: str, batas: int = 5) -> str:
    """Tarik berita terkini Indonesia dari Google News RSS apabila belum tersedia konteks dari website."""
    q_low = (pertanyaan or "").lower()
    kata_kunci_pemerintah = (
        "presiden", "wakil presiden", "wapres", "prabowo", "gibran",
        "menteri", "kabinet", "dpr", "mpr", "dpd", "mk", "ma", "kpk",
        "kejagung", "bpk", "pemerintah", "ikn", "mbg", "subsidi", "ppn",
        "bpjs", "indonesia", "berita", "kabar", "isu",
    )
    if not any(k in q_low for k in kata_kunci_pemerintah):
        return ""

    if "wakil presiden" in q_low or "wapres" in q_low or "gibran" in q_low:
        query = "Wakil Presiden Gibran Rakabuming Raka Indonesia"
    elif "presiden" in q_low or "prabowo" in q_low:
        query = "Presiden Prabowo Subianto pemerintah Indonesia"
    else:
        query = f"{pertanyaan.strip()[:120]} Indonesia"

    try:
        rss_url = f"https://news.google.com/rss/search?q={urllib.parse.quote(query)}&hl=id&gl=ID&ceid=ID:id"
        feed = feedparser.parse(rss_url)
        hasil = []
        for entry in (feed.entries or [])[:batas]:
            judul = (entry.get("title") or "").strip()
            sumber = entry.get("source", {}).get("title", "Media Indonesia")
            if judul:
                hasil.append(f"- {judul} ({sumber})")
        if not hasil:
            return ""
        return "Referensi Berita Terbaru Google News Indonesia:\n" + "\n".join(hasil)
    except Exception:
        return ""


def ringkas_data_live_sospem() -> str:
    """Ambil ringkasan real-time dari stream_analisis_sospem.json agar Chatbot tahu statistik terbaru."""
    if not os.path.exists("stream_analisis_sospem.json"):
        return "Data pemantauan lokal sedang dikumpulkan oleh agen SOSPEM."
    try:
        with open("stream_analisis_sospem.json", encoding="utf-8") as f:
            data = json.load(f)
        relevan = [d for d in data if d.get("Relevan")]
        if not relevan:
            return f"Total sampel terpantau: {len(data)} komentar."
        pos = sum(1 for d in relevan if d.get("Sentimen Publik") == "Positif")
        neg = sum(1 for d in relevan if d.get("Sentimen Publik") == "Negatif")
        net = len(relevan) - pos - neg
        bot = sum(1 for d in relevan if float(d.get("Skor Bot (%)", 0)) >= 60)
        warga = len(relevan) - bot
        lembaga_cnt = Counter()
        for d in relevan:
            for item_l in (d.get("Sentimen JSON") or []):
                if isinstance(item_l, dict) and item_l.get("lembaga"):
                    lembaga_cnt[f"{item_l['lembaga']} ({item_l.get('sentimen', 'Netral')})"] += 1
        top_lembaga = ", ".join([f"{k}: {v}" for k, v in lembaga_cnt.most_common(5)])
        terbaru = relevan[-1]
        return (
            f"Statistik File Notebook SOSPEM: Total opini pemerintahan={len(relevan)} (Warga Organik={warga}, Indikasi Bot={bot}; "
            f"Positif={pos}, Negatif={neg}, Netral={net}). "
            f"Sorotan Lembaga Teratas: {top_lembaga}. "
            f"Opini Terbaru dari {terbaru.get('Akun')} tentang '{terbaru.get('Sumber Berita')}': '{(terbaru.get('Komentar Asli') or '')[:120]}'."
        )
    except Exception:
        return "Statistik SOSPEM siap diakses."


def jawab_chat_sospem(messages_input: list, prompt_tunggal: str = "") -> str:
    """Fungsi utama penjawab Chatbot SOSPEM dengan prioritas konteks nyata & gaya teman bicara."""
    ringkasan_db = ringkas_data_live_sospem()
    persona_sistem = (
        "Kamu adalah Asisten AI SOSPEM (Sistem Pemantauan Opini Publik & Jaringan Bot) Indonesia, "
        "sekaligus teman bicara yang hangat, cerdas, jujur, dan luwes. "
        "Pemerintahan Republik Indonesia saat ini dipimpin oleh Presiden Prabowo Subianto dan Wakil Presiden Gibran Rakabuming Raka beserta Kabinet Merah Putih, DPR RI, MPR, DPD, MK, MA, KPK, Kejaksaan Agung, BPK, Komdigi, BSSN, dan Pemda. "
        f"Konteks Data Lokal Notebook SOSPEM: {ringkasan_db} "
        "ATURAN WAJIB AKURASI:\n"
        "1. Prioritaskan Konteks Tambahan dari Website SOSPEM (statistik PostgreSQL, berita Google News RSS terbaru, cuaca) sebagai rujukan fakta utama.\n"
        "2. Jangan pernah mengarang berita, nama tokoh asing yang tidak relevan, atau angka statistik palsu. Jika ditanya tentang 'presiden', 'wakil presiden', 'wapres', 'menteri', 'dpr', atau 'pemerintah kita', selalu bahas konteks Negara Kesatuan Republik Indonesia.\n"
        "3. Jika informasi terbaru tentang topik yang ditanyakan tidak ada di konteks, sampaikan secara jujur dan singkat lalu fokus pada data SOSPEM yang tersedia.\n"
        "4. Jawab dalam bahasa Indonesia yang natural dan mengalir tanpa memakai simbol bintang (** atau *) maupun tanda pagar (#), dan jangan pernah mencetak menu pilihan yang kaku."
    )

    chat_history = [{"role": "system", "content": persona_sistem}]
    ada_konteks_website = False
    pertanyaan_terakhir = prompt_tunggal.strip()

    if messages_input:
        for m in messages_input:
            role = m.get("role", "user")
            content = (m.get("content") or "").strip()
            if not content:
                continue
            if role == "system":
                ada_konteks_website = True
                chat_history[0]["content"] += "\n\nKonteks Tambahan dari Website SOSPEM:\n" + content
            else:
                if role == "user":
                    pertanyaan_terakhir = content
                chat_history.append({"role": role, "content": content})
    elif prompt_tunggal:
        chat_history.append({"role": "user", "content": prompt_tunggal.strip()})
    else:
        return "Halo! Ada yang ingin kita obrolkan atau analisis hari ini?"

    if not ada_konteks_website and pertanyaan_terakhir:
        rss_tambahan = cari_berita_rss_indonesia(pertanyaan_terakhir)
        if rss_tambahan:
            chat_history[0]["content"] += "\n\n" + rss_tambahan

    text_in = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_in], return_tensors="pt").to(model.device)
    with MODEL_LOCK, torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=380,
            temperature=0.25,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    balasan_mentah = tokenizer.decode(gen[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    return bersihkan_format_chat(balasan_mentah)


print(f"[SIAP] Mesin Scraper & Mesin Chatbot SOSPEM aktif! Target Website: {BASE_URL}")
print("Contoh Balasan Chatbot:", jawab_chat_sospem([{"role": "user", "content": "Halo, apa kabar pantauan opini pemerintah hari ini?"}]))

### Sel 4: Nyalakan Server API Chatbot (`Port 11434`) & Reverse-Bridge ke Website SOSPEM (`https://sospem.pubhe.cloud`)
Tekan **`Shift + Enter`**. Sel ini menyalakan dua layanan latar belakang secara bersamaan:
1. **Server FastAPI Lokal (`Port 11434`)** dengan endpoint `POST /api/chat`, `POST /api/generate`, dan `POST /v1/chat/completions`.
2. **Thread Reverse-Polling Bridge** yang memantau `{BASE_URL}/api/chat-bridge/pending` setiap 2 detik, memproses pertanyaan user website memakai `Qwen/Qwen2.5-1.5B-Instruct`, dan mengirimkan balasannya ke `POST {BASE_URL}/api/chat-bridge/{task_id}/reply` sehingga AI di JupyterHub terhubung langsung dengan website tanpa perlu port publik atau Cloudflare Tunnel.

In [ ]:
import os, time, threading, requests, uvicorn
from fastapi import FastAPI, Request
from datetime import datetime, timezone

BASE_URL = (globals().get("BASE_URL") or os.environ.get("SOSPEM_BASE_URL", "https://sospem.pubhe.cloud")).rstrip("/")
SOSPEM_API_KEY = globals().get("SOSPEM_API_KEY") if "SOSPEM_API_KEY" in globals() else os.environ.get("SOSPEM_API_KEY", "")
HEADERS_LARAVEL = {"X-Sospem-Key": SOSPEM_API_KEY} if SOSPEM_API_KEY else {}

chat_app = FastAPI(title="SOSPEM Local AI Chatbot Server")


@chat_app.get("/")
@chat_app.get("/api/tags")
async def status_server():
    return {
        "status": "online",
        "models": [{"name": "qwen2.5:1.5b"}, {"name": "qwen2.5:3b"}],
        "service": "SOSPEM AI Chatbot & Government Analyst",
        "bridge_target": BASE_URL,
    }


@chat_app.post("/api/chat")
@chat_app.post("/api/generate")
async def ollama_compatible_chat(req: Request):
    body = await req.json()
    msgs = body.get("messages") or []
    prompt = body.get("prompt") or ""
    balasan = jawab_chat_sospem(msgs, prompt)
    return {
        "model": body.get("model", "qwen2.5:1.5b"),
        "created_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "message": {"role": "assistant", "content": balasan},
        "response": balasan,
        "done": True,
    }


@chat_app.post("/v1/chat/completions")
async def openai_compatible_chat(req: Request):
    body = await req.json()
    msgs = body.get("messages") or []
    balasan = jawab_chat_sospem(msgs)
    return {
        "id": "chatcmpl-sospem",
        "object": "chat.completion",
        "choices": [{"index": 0, "message": {"role": "assistant", "content": balasan}, "finish_reason": "stop"}],
    }


def _jalankan_server_chatbot():
    try:
        uvicorn.run(chat_app, host="0.0.0.0", port=11434, log_level="warning")
    except Exception as e:
        print(f"[Info Server Chatbot] Port 11434 sudah aktif atau digunakan ({e})")


def _jalankan_bridge_chatbot_laravel():
    print(f"[BRIDGE CHAT] Reverse-polling aktif menuju {BASE_URL}/api/chat-bridge/pending ...")
    last_err_at = 0.0
    while True:
        try:
            target_base = (globals().get("BASE_URL") or BASE_URL).rstrip("/")
            if not target_base:
                time.sleep(3)
                continue

            resp = requests.get(
                f"{target_base}/api/chat-bridge/pending",
                headers=HEADERS_LARAVEL,
                timeout=10,
            )
            if resp.ok:
                data = resp.json()
                task = data.get("task") if isinstance(data, dict) else None
                if isinstance(task, dict) and task.get("id") and task.get("messages"):
                    task_id = str(task["id"])
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] [BRIDGE CHAT] Memproses pertanyaan website #{task_id[:8]} via Qwen2.5...")
                    balasan = jawab_chat_sospem(task["messages"], "")
                    kirim = requests.post(
                        f"{target_base}/api/chat-bridge/{task_id}/reply",
                        json={"reply": balasan, "model": globals().get("MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")},
                        headers=HEADERS_LARAVEL,
                        timeout=15,
                    )
                    if kirim.ok:
                        print(f"[{datetime.now().strftime('%H:%M:%S')}] [BRIDGE CHAT] Balasan #{task_id[:8]} terkirim ke website.")
                    else:
                        print(f"[BRIDGE CHAT] Gagal mengirim balasan #{task_id[:8]}: HTTP {kirim.status_code}")
                    continue

            time.sleep(2)
        except Exception as e:
            now_ts = time.time()
            if now_ts - last_err_at >= 30:
                print(f"[Info Bridge Chat] Gangguan koneksi sementara ({e}), mencoba ulang...")
                last_err_at = now_ts
            time.sleep(3)


if not globals().get("CHATBOT_SERVER_STARTED"):
    t_chat = threading.Thread(target=_jalankan_server_chatbot, daemon=True)
    t_chat.start()
    CHATBOT_SERVER_STARTED = True
    print("[AKTIF] Server AI Chatbot SOSPEM berjalan di latar belakang pada http://127.0.0.1:11434/api/chat !")
else:
    print("[AKTIF] Server AI Chatbot SOSPEM sudah berjalan di latar belakang!")

if not globals().get("CHATBOT_BRIDGE_STARTED"):
    t_bridge = threading.Thread(target=_jalankan_bridge_chatbot_laravel, daemon=True)
    t_bridge.start()
    CHATBOT_BRIDGE_STARTED = True
    print(f"[AKTIF] Reverse-Bridge Chatbot berjalan di latar belakang menuju {BASE_URL} !")
else:
    print("[AKTIF] Reverse-Bridge Chatbot sudah berjalan di latar belakang!")

### Sel 5 (Opsional): Daftarkan Akun Riset X / Twitter Mandiri (`twscrape`)
Lewati sel ini jika tidak memakai akun riset X.

In [ ]:
import asyncio, nest_asyncio
from twscrape import API

nest_asyncio.apply()

X_USER = ""
X_PASS = ""
X_EMAIL = ""
X_EMAIL_PASS = ""
X_COOKIES = ""

async def daftarkan_akun_x():
    if not X_USER:
        print("[INFO] Akun X mandiri belum diisi. Lewati sel ini.")
        return
    api = API("sospem_x_accounts.db")
    if X_COOKIES:
        await api.pool.add_account(X_USER, X_PASS or "dummy", X_EMAIL or "dummy@mail.com", X_EMAIL_PASS or "dummy", cookies=X_COOKIES)
    else:
        await api.pool.add_account(X_USER, X_PASS, X_EMAIL, X_EMAIL_PASS)
        await api.pool.login_all()
    print("[BERHASIL] Akun X mandiri terdaftar di sospem_x_accounts.db!")

asyncio.run(daftarkan_akun_x())

### Sel 6: Jalankan Multi-Channel Scraper Agent 24/7 + Live Sync ke Website SOSPEM
Tekan **`Shift + Enter`**. Selagi sel ini memutar siklus penarikan data & memantau antrean `Scrape Live` website (`https://sospem.pubhe.cloud`), **Server AI Chatbot & Reverse-Bridge di Sel 4 tetap aktif di latar belakang** melayani percakapan pengguna!

In [ ]:
import os, re, time, json, random, asyncio, urllib.parse, requests, hashlib
import nest_asyncio
import pandas as pd
from datetime import datetime
from itertools import islice
import yt_dlp
import feedparser
import instaloader
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_RECENT

nest_asyncio.apply()

# ==============================================================================
# 1. KONFIGURASI UTAMA (DOMAIN WEBSITE SOSPEM & KREDENSIAL OPSIONAL)
# ==============================================================================
BASE_URL = os.environ.get("SOSPEM_BASE_URL", "https://sospem.pubhe.cloud").rstrip("/")

IG_USERNAME_RISET = os.environ.get("IG_USERNAME_RISET", "")
IG_PASSWORD_RISET = os.environ.get("IG_PASSWORD_RISET", "")
TIKTOK_MS_TOKEN = os.environ.get("TIKTOK_MS_TOKEN", "")

APIFY_API_TOKEN = os.environ.get("APIFY_API_TOKEN", "")
APIFY_HEADERS = {"Authorization": f"Bearer {APIFY_API_TOKEN}"}

SOSPEM_API_KEY = os.environ.get("SOSPEM_API_KEY", "")
HEADERS_LARAVEL = {"X-Sospem-Key": SOSPEM_API_KEY} if SOSPEM_API_KEY else {}

PLATFORM_DIDUKUNG_LARAVEL = {"x", "tiktok", "meta", "youtube", "news"}
PETA_SLOT_LARAVEL = {"meta": "meta", "x": "x", "tiktok": "tiktok", "youtube": "tiktok", "news": "news"}
PETA_SENTIMEN_LARAVEL = {"Positif": "positive", "Negatif": "negative", "Netral": "neutral"}

AKUN_IG_BERITA = ["kumparancom", "detikcom", "kompascom", "tempodotco", "cnnindonesia", "narasi.tv"]
AKUN_TIKTOK_BERITA = ["kompastv", "kumparan", "detikcom", "tempodotco"]

yt_downloader = YoutubeCommentDownloader()


def jeda_manusia(min_detik=1.0, max_detik=2.5):
    time.sleep(random.uniform(min_detik, max_detik))


def parse_likes_aman(val) -> int:
    if isinstance(val, (int, float)):
        return int(val)
    s = str(val or "0").strip().upper()
    try:
        if "K" in s or "RB" in s:
            num = float(re.sub(r"[^0-9.]", "", s.replace(",", ".")) or 0)
            return int(num * 1000)
        if "M" in s or "JT" in s:
            num = float(re.sub(r"[^0-9.]", "", s.replace(",", ".")) or 0)
            return int(num * 1000000)
        return int(re.sub(r"[^0-9]", "", s) or 0)
    except Exception:
        return 0


def tarik_youtube_by_tema(tema: str, id_terproses: set, target_jumlah: int) -> list:
    hasil_item = []
    ydl_opts = {"quiet": True, "extract_flat": True, "skip_download": True}
    video_map = {}
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(f"ytsearch3:{tema} indonesia", download=False)
            for e in (info.get("entries") or []):
                if e and e.get("id") and e.get("title"):
                    video_map[e["id"]] = {
                        "url": f"https://www.youtube.com/watch?v={e['id']}",
                        "judul": e["title"],
                        "konteks": f"Video Berita {e.get('uploader') or 'Media'} ({tema}): {e['title']}",
                    }
        for vid, meta in video_map.items():
            if len(hasil_item) >= target_jumlah:
                break
            try:
                gen = yt_downloader.get_comments_from_url(meta["url"], sort_by=SORT_BY_RECENT)
                for c in islice(gen, target_jumlah * 2):
                    if len(hasil_item) >= target_jumlah:
                        break
                    teks = (c.get("text") or "").strip()
                    user = (c.get("author") or "warganet").lstrip("@")
                    cid = f"vid_{c.get('cid') or f'{vid}_{user}_{teks[:20]}'}"
                    if teks and cid not in id_terproses:
                        hasil_item.append({
                            "cid": cid, "platform": "youtube", "kanal": f"YouTube ({tema})",
                            "username": user, "text": teks,
                            "likes": parse_likes_aman(c.get("votes")),
                            "url": meta["url"], "judul_berita": meta["judul"], "konteks": meta["konteks"],
                        })
                jeda_manusia(1.0, 2.0)
            except Exception:
                continue
    except Exception as e:
        print(f"   [Info YouTube] {e}")
    return hasil_item


def tarik_rss_by_tema(tema: str, id_terproses: set, target_jumlah: int) -> list:
    hasil_item = []
    if target_jumlah <= 0:
        return hasil_item
    try:
        q_enc = urllib.parse.quote(tema)
        rss_url = f"https://news.google.com/rss/search?q={q_enc}&hl=id&gl=ID&ceid=ID:id"
        feed = feedparser.parse(rss_url)
        for entry in feed.entries[:target_jumlah * 2]:
            if len(hasil_item) >= target_jumlah:
                break
            judul = (entry.get("title") or "").strip()
            link = entry.get("link") or ""
            sumber = entry.get("source", {}).get("title", "Media Nasional")
            cid = "rss_" + hashlib.md5((link or judul).encode("utf-8")).hexdigest()[:16]
            if judul and cid not in id_terproses:
                hasil_item.append({
                    "cid": cid, "platform": "news", "kanal": f"Google News RSS ({tema})",
                    "username": sumber.replace(" ", "_").lower(),
                    "text": judul, "likes": 0,
                    "url": link, "judul_berita": judul, "konteks": f"Tajuk Berita {sumber} ({tema}): {judul}",
                })
    except Exception as e:
        print(f"   [Info RSS] {e}")
    return hasil_item


# ==============================================================================
# FUNGSI PENDUKUNG: PAYLOAD DINAMIS PER TEMA, KIRIM, & SIMPAN LOKAL
# ==============================================================================
FILE_JSON = "stream_analisis_sospem.json"
FILE_EXCEL = "stream_analisis_sospem.xlsx"
FILE_IDS = "komentar_terproses_ids.json"


def normalisasi_teks(t: str) -> str:
    return re.sub(r"\W+", " ", (t or "").lower()).strip()


def bangun_payload(baris: dict, tema_aktif: str = "auto") -> dict:
    plat_asli = baris["Platform SOSPEM"]
    plat_laravel = PETA_SLOT_LARAVEL.get(plat_asli, "meta")
    slug_dinamis = re.sub(r"[^a-z0-9]+", "-", tema_aktif.lower()).strip("-") if tema_aktif and tema_aktif != "auto" else "auto"

    return {
        "issue_slug": slug_dinamis,
        "issue_title": tema_aktif,
        "platform": plat_laravel,
        "platform_origin": plat_asli,
        "source_channel": baris.get("Kanal"),
        "external_id": baris.get("cid"),
        "author_name": baris["Akun"].lstrip("@"),
        "author_handle": baris["Akun"],
        "content": baris["Komentar Asli"],
        "likes_count": baris["Likes"],
        "source_url": baris["URL"],
        "source_news": baris["Sumber Berita"],
        "ai_sentiment": PETA_SENTIMEN_LARAVEL.get(baris["Sentimen Publik"], "neutral"),
        "ai_sentiment_label": baris["Sentimen Publik"],
        "ai_bot_score": baris["Skor Bot (%)"],
        "ai_bot_reason": baris["Catatan Forensik SOSPEM"],
        "ai_institutions": baris["Sentimen JSON"],
    }


def kirim_baris(baris: dict, tema_aktif: str = "auto") -> str:
    if not BASE_URL:
        return "offline"
    if baris.get("Platform SOSPEM") not in PLATFORM_DIDUKUNG_LARAVEL:
        return "dilewati"
    try:
        resp = requests.post(
            f"{BASE_URL.rstrip('/')}/api/ingest",
            json=bangun_payload(baris, tema_aktif),
            headers=HEADERS_LARAVEL,
            timeout=15,
        )
        if resp.ok:
            baris["Terkirim"] = True
            try:
                return resp.json().get("status", f"HTTP {resp.status_code}")
            except Exception:
                return f"HTTP {resp.status_code}"
        if 400 <= resp.status_code < 500 and resp.status_code not in (408, 429):
            baris["Terkirim"] = "ditolak"
        return f"ERR {resp.status_code}"
    except Exception:
        return "KONEKSI_ERR"


def simpan_semua(tulis_excel=True):
    with open(FILE_JSON, "w", encoding="utf-8") as f:
        json.dump(semua_hasil, f, indent=2, ensure_ascii=False)
    with open(FILE_IDS, "w", encoding="utf-8") as f:
        json.dump(list(id_terproses), f)
    if tulis_excel and semua_hasil:
        try:
            df_x = pd.DataFrame(semua_hasil)
            if "Sentimen JSON" in df_x.columns:
                df_x["Sentimen JSON"] = df_x["Sentimen JSON"].apply(lambda v: json.dumps(v, ensure_ascii=False))
            df_x.to_excel(FILE_EXCEL, index=False)
        except Exception as e:
            print(f"   [Info Excel] Gagal menulis Excel ({e}). JSON tetap aman.")


if os.path.exists(FILE_JSON):
    with open(FILE_JSON, encoding="utf-8") as f:
        semua_hasil = json.load(f)
else:
    semua_hasil = []

if os.path.exists(FILE_IDS):
    with open(FILE_IDS, encoding="utf-8") as f:
        id_terproses = set(json.load(f))
else:
    id_terproses = set()

teks_terlihat = {}
for _b in semua_hasil:
    _k = normalisasi_teks(_b.get("Komentar Asli"))
    if len(_k) >= 15:
        teks_terlihat.setdefault(_k, set()).add(_b.get("Akun", ""))

print("=================================================================")
print(" SOSPEM ON-DEMAND LIVE SCRAPER + CHATBOT SERVER AKTIF (v5.1) ")
print("=================================================================")
print(f"Target Website SOSPEM : {BASE_URL if BASE_URL else 'BELUM DIISI'}")
print("Server AI Chatbot     : http://127.0.0.1:11434/api/chat + Reverse-Bridge (Aktif)")
print("Status Listener       : Siap menerima perintah dari tombol 'Scrape Live' di Website...\n")

# ==============================================================================
# LOOP LISTENER ON-DEMAND (MENGECEK TOMBOL WEBSITE TIAP 5 DETIK)
# ==============================================================================
while True:
    try:
        if not BASE_URL:
            time.sleep(5)
            continue

        cek = requests.get(f"{BASE_URL.rstrip('/')}/api/scrape-jobs/pending", headers=HEADERS_LARAVEL, timeout=10)
        job = cek.json().get("job") if cek.ok else None

        if not job:
            time.sleep(5)
            continue

        tema = str(job.get("topic") or "Pemerintahan").strip()
        batas = max(1, min(50, int(job.get("limit") or 10)))
        print(f"\n[{datetime.now().strftime('%H:%M:%S')}] [PERINTAH WEB #{job['id']}] Menarik {batas} data bertema: '{tema}'...")

        target_yt = max(1, int(batas * 0.7))
        yt_items = tarik_youtube_by_tema(tema, id_terproses, target_jumlah=target_yt)
        sisa_target = max(0, batas - len(yt_items))
        rss_items = tarik_rss_by_tema(tema, id_terproses, target_jumlah=sisa_target)

        antrean_baru = list({it["cid"]: it for it in (yt_items + rss_items)}.values())[:batas]
        print(f"   Ditemukan {len(antrean_baru)} item baru ({len(yt_items)} YouTube, {len(rss_items)} News). Memulai analisis AI Qwen...")

        terkirim_sukses = 0
        for idx, item in enumerate(antrean_baru, start=1):
            hasil_ai = analisis_komentar_kontekstual(
                item["username"], item["text"], item["likes"], item["konteks"]
            )

            if item["platform"] == "news":
                hasil_ai.indikasi_bot_persen = 0.0
                hasil_ai.alasan_indikasi_bot = "Tajuk berita (bukan komentar akun)"
            else:
                kunci = normalisasi_teks(item["text"])
                if len(kunci) >= 15:
                    akun_set = teks_terlihat.setdefault(kunci, set())
                    akun_set.add(f"@{item['username']}")
                    if len(akun_set) >= 3:
                        hasil_ai.indikasi_bot_persen = max(hasil_ai.indikasi_bot_persen, 85.0)
                        hasil_ai.alasan_indikasi_bot = (
                            f"Teks identik dipakai {len(akun_set)} akun berbeda | " + hasil_ai.alasan_indikasi_bot
                        )

            relevan_bool = bool(hasil_ai.relevan_pemerintahan)
            daftar_lembaga_dict = [x.model_dump() for x in hasil_ai.sentimen_per_lembaga]
            rincian = " | ".join([f"{x.lembaga}: {x.sentimen} ({x.alasan})" for x in hasil_ai.sentimen_per_lembaga]) or "Tidak Ada (Noise)"

            baris = {
                "cid": item["cid"],
                "Waktu": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "Platform SOSPEM": item["platform"],
                "Kanal": item["kanal"],
                "Sumber Berita": item["judul_berita"][:120],
                "URL": item["url"],
                "Akun": f"@{item['username']}",
                "Komentar Asli": item["text"],
                "Likes": item["likes"],
                "Relevan": relevan_bool,
                "Relevan Pemerintahan?": f"{'Ya' if relevan_bool else 'Tidak'} ({relevan_bool})",
                "Sentimen Publik": hasil_ai.sentimen_publik,
                "Rincian Sentimen per Lembaga (Terpisah)": rincian,
                "Sentimen JSON": daftar_lembaga_dict,
                "Skor Bot (%)": hasil_ai.indikasi_bot_persen,
                "Catatan Forensik SOSPEM": hasil_ai.alasan_indikasi_bot,
                "Terkirim": False,
            }

            status_web = kirim_baris(baris, tema_aktif=tema) if relevan_bool else "tidak_relevan"
            if baris["Terkirim"] is True:
                terkirim_sukses += 1

            semua_hasil.append(baris)
            id_terproses.add(item["cid"])
            print(f"   [{idx}/{len(antrean_baru)}] [{item['platform'].upper()}] @{item['username']}: \"{item['text'][:40]}...\" -> {hasil_ai.sentimen_publik} | Relevan: {relevan_bool} | Web: {status_web}")

        if antrean_baru:
            simpan_semua(tulis_excel=True)

        requests.post(
            f"{BASE_URL.rstrip('/')}/api/scrape-jobs/{job['id']}/done",
            json={"scraped_count": terkirim_sukses},
            headers=HEADERS_LARAVEL,
            timeout=10,
        )
        print(f"=== [SELESAI] Tugas #{job['id']} ('{tema}'): {terkirim_sukses}/{len(antrean_baru)} data berhasil masuk ke database! ===")

    except KeyboardInterrupt:
        print("\n[BERHENTI] Listener dihentikan oleh pengguna (Server Chatbot & Bridge di Sel 4 tetap menyala).")
        simpan_semua(tulis_excel=True)
        break
    except Exception as e:
        print(f"[INFO] Gangguan sementara ({e}), mencoba ulang dalam 5 detik...")
        time.sleep(5)